# Demo Streaming Replay cho hệ thống phát hiện thao túng thị trường

Notebook này dùng để demo luồng streaming: khởi động Redis, chạy Spark Streaming ở background, replay dữ liệu vào Redis Stream, lưu log và lọc các dòng kết quả quan trọng. Code chính của project không nằm trong notebook; notebook chỉ điều khiển và ghi lại output demo


## 1. Thiết lập môi trường

Đặt thư mục project, cấu hình Python/PySpark/Hadoop và kiểm tra các file quan trọng trước khi chạy demo


In [2]:
import os
import sys
import time
import shutil
import subprocess
from pathlib import Path
from collections import deque

PROJECT_ROOT = Path(r"D:\BigDataSang")
os.chdir(PROJECT_ROOT)

os.environ["HADOOP_HOME"] = str(PROJECT_ROOT / "hadoop")
os.environ["PATH"] = os.environ["HADOOP_HOME"] + r"\bin;" + os.environ["PATH"]

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

Path("outputs").mkdir(exist_ok=True)

print("Project root:", Path.cwd())
print("Python:", sys.executable)
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])

required_files = [
    "docker-compose.yml",
    "src/python/streaming_engine.py",
    "src/python/replay_manipulation.py",
    "data/replay/test_spoofing_replay.json",
    "models/spoofing_rf_model.pkl",
]

for f in required_files:
    print(f, "OK" if Path(f).exists() else "MISSING")

Project root: D:\BigDataSang
Python: c:\Users\Mr Long\AppData\Local\Programs\Python\Python311\python.exe
HADOOP_HOME: D:\BigDataSang\hadoop
docker-compose.yml OK
src/python/streaming_engine.py OK
src/python/replay_manipulation.py OK
data/replay/test_spoofing_replay.json OK
models/spoofing_rf_model.pkl OK


## 2. Khởi động Redis

Redis Stream được dùng làm nơi nhận dữ liệu replay trước khi Spark Structured Streaming đọc theo micro-batch


In [3]:
result = subprocess.run(
    ["docker", "compose", "up", "-d", "redis"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)

if result.returncode == 0:
    print("Redis started.")
else:
    print("Redis start failed.")


time="2026-06-21T17:46:59+07:00" level=warning msg="D:\\BigDataSang\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Container redis Running 

Redis started.


## 3. Dọn dữ liệu và log cũ

Xóa log demo cũ, xóa Redis Stream cũ và checkpoint cũ để lần chạy mới không bị lẫn dữ liệu từ lần trước


In [4]:
for log_file in [
    Path("outputs/streaming_demo_log.txt"),
    Path("outputs/replay_demo_log.txt")
]:
    if log_file.exists():
        log_file.unlink()
        print("Deleted old log:", log_file)

result = subprocess.run(
    [
        sys.executable,
        "-c",
        "import redis; r=redis.Redis(host='localhost', port=6379, decode_responses=True); r.delete('binance:depth'); print('deleted binance:depth')"
    ],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)

checkpoint_paths = [
    Path(r"\tmp\binance_redis_checkpoint"),
    Path(r"D:\tmp\binance_redis_checkpoint"),
    PROJECT_ROOT / "tmp" / "binance_redis_checkpoint",
]

for p in checkpoint_paths:
    if p.exists():
        print("Removing checkpoint:", p)
        shutil.rmtree(p, ignore_errors=True)

print("Cleanup done.")

Deleted old log: outputs\streaming_demo_log.txt
Deleted old log: outputs\replay_demo_log.txt
deleted binance:depth


Removing checkpoint: \tmp\binance_redis_checkpoint
Cleanup done.


## 4. Chạy Spark Streaming Engine

Chạy `streaming_engine.py` ở background và ghi toàn bộ log streaming vào `outputs/streaming_demo_log.txt`


In [5]:
streaming_log_path = PROJECT_ROOT / "outputs" / "streaming_demo_log.txt"

streaming_log_file = open(streaming_log_path, "w", encoding="utf-8", errors="replace")

streaming_proc = subprocess.Popen(
    [sys.executable, "src/python/streaming_engine.py"],
    cwd=PROJECT_ROOT,
    stdout=streaming_log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
    text=True
)

print("Started streaming_engine.py")
print("PID:", streaming_proc.pid)
print("Log file:", streaming_log_path)

time.sleep(20)

if streaming_proc.poll() is None:
    print("Streaming engine is running.")
else:
    print("Streaming engine stopped early. Check log.")

Started streaming_engine.py
PID: 18124
Log file: D:\BigDataSang\outputs\streaming_demo_log.txt
Streaming engine is running.


## 5. Kiểm tra streaming đã sẵn sàng

Đọc phần cuối log streaming để kiểm tra model đã được load và Spark đã bắt đầu chờ Redis Stream


In [6]:
def tail_file(path, n=80):
    path = Path(path)
    if not path.exists():
        print("File not found:", path)
        return
    
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = deque(f, maxlen=n)
    
    print("".join(lines))

tail_file("outputs/streaming_demo_log.txt", 80)

2026-06-21 17:47:13,086 [INFO] Đã tự động tạo consumer group 'spark-source' cho stream 'binance:depth' trên Redis.
2026-06-21 17:47:13,088 [INFO] Khởi động Spark Structured Streaming Engine (Redis Streams)...
Missing Python executable 'c:\Users\Mr Long\AppData\Local\Programs\Python\Python311\python.exe', defaulting to 'C:\Users\Mr Long\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\bin\..' for SPARK_HOME environment variable. Please install Python or specify the correct Python executable in PYSPARK_DRIVER_PYTHON or PYSPARK_PYTHON environment variable to detect SPARK_HOME safely.
:: loading settings :: url = jar:file:/C:/Users/Mr%20Long/AppData/Local/Programs/Python/Python311/Lib/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: C:\Users\Mr Long\.ivy2\cache
The jars for the packages stored in: C:\Users\Mr Long\.ivy2\jars
com.redislabs#spark-redis_2.12 added as a dependency
:: resolving dependencies :: org.

## 6. Replay dữ liệu vào Redis Stream

Chạy `replay_manipulation.py` để đẩy file `test_spoofing_replay.json` vào Redis. Spark sẽ đọc các event này và xử lý theo batch


In [7]:
replay_log_path = PROJECT_ROOT / "outputs" / "replay_demo_log.txt"

with open(replay_log_path, "w", encoding="utf-8", errors="replace") as replay_log_file:
    result = subprocess.run(
        [
            sys.executable,
            "src/python/replay_manipulation.py",
            "--file",
            "data/replay/test_spoofing_replay.json",
            "--sleep",
            "0.1"
        ],
        cwd=PROJECT_ROOT,
        stdout=replay_log_file,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
        text=True
    )

print("Replay finished.")
print("Return code:", result.returncode)
print("Replay log:", replay_log_path)

time.sleep(30)
print("Waited for Spark to process remaining batches.")

Replay finished.
Return code: 0
Replay log: D:\BigDataSang\outputs\replay_demo_log.txt
Waited for Spark to process remaining batches.


## 7. Dừng Spark Streaming Engine

Sau khi replay xong và Spark đã xử lý đủ batch, dừng tiến trình streaming đang chạy ở background


In [8]:
if streaming_proc.poll() is None:
    print("Stopping streaming engine...")
    streaming_proc.terminate()
    
    try:
        streaming_proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        print("Force killing streaming engine...")
        streaming_proc.kill()

streaming_log_file.close()

print("Streaming stopped.")

Stopping streaming engine...
Streaming stopped.


## 8. Xem log replay

Hiển thị log của quá trình replay dữ liệu vào Redis Stream


In [9]:
tail_file("outputs/replay_demo_log.txt", 80)

2026-06-21 18:04:39,868 [INFO] Event #9922/10000 pushing... timestamp=1718800992100 bids=10 asks=10
2026-06-21 18:04:39,970 [INFO] Event #9923/10000 pushing... timestamp=1718800992200 bids=10 asks=10
2026-06-21 18:04:40,073 [INFO] Event #9924/10000 pushing... timestamp=1718800992300 bids=10 asks=10
2026-06-21 18:04:40,175 [INFO] Event #9925/10000 pushing... timestamp=1718800992400 bids=10 asks=10
2026-06-21 18:04:40,277 [INFO] Event #9926/10000 pushing... timestamp=1718800992500 bids=10 asks=10
2026-06-21 18:04:40,380 [INFO] Event #9927/10000 pushing... timestamp=1718800992600 bids=10 asks=10
2026-06-21 18:04:40,482 [INFO] Event #9928/10000 pushing... timestamp=1718800992700 bids=10 asks=10
2026-06-21 18:04:40,585 [INFO] Event #9929/10000 pushing... timestamp=1718800992800 bids=10 asks=10
2026-06-21 18:04:40,687 [INFO] Event #9930/10000 pushing... timestamp=1718800992900 bids=10 asks=10
2026-06-21 18:04:40,790 [INFO] Event #9931/10000 pushing... timestamp=1718800993000 bids=10 asks=10


## 9. Xem log streaming đầy đủ

Hiển thị phần cuối của log streaming, gồm LOB snapshot, ML probability, cảnh báo và kết quả xử lý batch nếu có


In [10]:
tail_file("outputs/streaming_demo_log.txt", 160)


[Stage 176:>                                                      (0 + 12) / 12]

                                                                                
2026-06-21 18:03:29,816 [INFO] Xác suất thao túng (ML): 49.74%
2026-06-21 18:03:29,816 [INFO] Thị trường bình thường.
2026-06-21 18:03:29,816 [INFO] --- Batch xử lý hoàn tất ---
2026-06-21 18:03:30,333 [INFO] Received command c on object id p0
2026-06-21 18:03:30,370 [INFO] --- Bắt đầu xử lý Batch ID: 67 (99 events) ---
2026-06-21 18:03:30,524 [INFO] LOB Snapshot -> Best Bid: 130.20 | Best Ask: 130.30 | Spread: 0.10
2026-06-21 18:03:30,539 [INFO] [Feature Extraction] spread=0.1000, bid_vol=19176000.00, ask_vol=6074000.00, imbalance=0.5189, cancel_rate=0.0071

[Stage 178:>                                                      (0 + 12) / 12]

[Stage 178:==================================================>    (11 + 1) / 12]

                                                                                
2026-06-21 18:03:39,906 [

## 10. Lọc các dòng log quan trọng

Đọc file `outputs/streaming_demo_log.txt` và chỉ giữ lại các dòng quan trọng trong quá trình demo

Mục đích là giúp tập trung vào các bước chính của pipeline: load model, nhận batch từ Redis Stream, cập nhật LOB, trích xuất đặc trưng, dự đoán xác suất thao túng và phát hiện cảnh báo nếu có

In [ ]:
important_keywords = [
    "Loaded trained model",
    "Listening on Redis Stream",
    "Bắt đầu xử lý Batch",
    "LOB Snapshot",
    "Feature Extraction",
    "Xác suất thao túng",
    "CẢNH BÁO",
    "Spark Distributed Alpha-Beta",
    "Avg Risk",
    "RED FLAG",
    "Batch xử lý hoàn tất"
]

log_path = Path("outputs/streaming_demo_log.txt")

with open(log_path, "r", encoding="utf-8", errors="replace") as f:
    lines = f.readlines()

filtered = []
for line in lines:
    if any(k in line for k in important_keywords):
        filtered.append(line)

print("".join(filtered[-200:]))

2026-06-21 17:56:37,245 [INFO] LOB Snapshot -> Best Bid: 128.40 | Best Ask: 128.50 | Spread: 0.10
2026-06-21 17:56:37,261 [INFO] [Feature Extraction] spread=0.1000, bid_vol=18002000.00, ask_vol=13725000.00, imbalance=0.1348, cancel_rate=0.0000
2026-06-21 17:56:46,817 [INFO] Xác suất thao túng (ML): 16.33%
2026-06-21 17:56:46,817 [INFO] --- Batch xử lý hoàn tất ---
2026-06-21 17:56:47,373 [INFO] --- Bắt đầu xử lý Batch ID: 37 (102 events) ---
2026-06-21 17:56:47,538 [INFO] LOB Snapshot -> Best Bid: 128.20 | Best Ask: 128.30 | Spread: 0.10
2026-06-21 17:56:47,558 [INFO] [Feature Extraction] spread=0.1000, bid_vol=18002000.00, ask_vol=13725000.00, imbalance=0.1348, cancel_rate=0.0161
2026-06-21 17:56:57,127 [INFO] Xác suất thao túng (ML): 27.33%
2026-06-21 17:56:57,127 [INFO] --- Batch xử lý hoàn tất ---
2026-06-21 17:56:57,702 [INFO] --- Bắt đầu xử lý Batch ID: 38 (100 events) ---
2026-06-21 17:56:57,842 [INFO] LOB Snapshot -> Best Bid: 128.00 | Best Ask: 128.10 | Spread: 0.10
2026-06-21